<a href="https://colab.research.google.com/github/Anoshafatima131/ML-Project/blob/main/Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
wisam1985_advanced_iot_agriculture_2024_path = kagglehub.dataset_download('wisam1985/advanced-iot-agriculture-2024')
anoshafatima131_advanced_iot_agriculture_path = kagglehub.notebook_output_download('anoshafatima131/advanced-iot-agriculture')

print('Data source import complete.')


IoT Agriculture ML Project — Phase 3: Feature Engineering

In this phase, we focus on improving model performance through feature engineering techniques. This includes:

Identifying important features using multiple algorithms
Creating new meaningful features
Evaluating feature performance
Applying scaling techniques
Exploring clustering-based feature creation

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

import lightgbm as lgb

# Optional (if you use them)
import shap
from lime.lime_tabular import LimeTabularExplainer

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/wisam1985/advanced-iot-agriculture-2024/Advanced_IoT_Dataset.csv")

df.head()

In [ ]:
df.info()
df.describe()
df.isnull().sum()

In [ ]:
df.columns

In [ ]:
df.columns = df.columns.str.strip()

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# Load dataset
df = pd.read_csv("/kaggle/input/datasets/wisam1985/advanced-iot-agriculture-2024/Advanced_IoT_Dataset.csv")

# Strip leading/trailing spaces from column names
df.columns = df.columns.str.strip()

# Check column names after stripping
print(df.columns)

# Drop useless column if it exists
if 'Random' in df.columns:
    df = df.drop('Random', axis=1)
else:
    print("'Random' column not found, skipping drop.")

# Define features and target
X = df.drop('Class', axis=1)
y = df['Class']

# Train Random Forest
rf = RandomForestClassifier()
rf.fit(X, y)

# Feature importance
rf_importance = pd.Series(rf.feature_importances_, index=X.columns)
rf_importance = rf_importance.sort_values(ascending=False)
print(rf_importance)

In [ ]:
dt = DecisionTreeClassifier()
dt.fit(X, y)

dt_importance = pd.Series(dt.feature_importances_, index=X.columns)

In [ ]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X, y)

lr_importance = pd.Series(abs(lr.coef_[0]), index=X.columns)

In [ ]:
lgb_model = lgb.LGBMClassifier()
lgb_model.fit(X, y)

lgb_importance = pd.Series(lgb_model.feature_importances_, index=X.columns)

In [ ]:
importance_df = pd.DataFrame({
    'RF': rf_importance,
    'DT': dt_importance,
    'LR': lr_importance,
    'LGB': lgb_importance
})

importance_df.fillna(0).mean(axis=1).sort_values(ascending=False)

Feature Importance Analysis

Feature importance was evaluated using multiple algorithms including Random Forest, Decision Tree, Logistic Regression, and LightGBM. The average importance score across these models was calculated to identify the most influential features.

Important Features

The following features were found to be highly important as they consistently showed high importance across multiple models:

Average of chlorophyll in the plant (ACHP)

Plant height rate (PHR)

Average leaf area of the plant (ALAP)

Average number of plant leaves (ANPL)

Average root length (ARL)

These features are directly related to plant growth and health. For example, chlorophyll content indicates photosynthesis efficiency, while plant height and leaf area reflect growth rate. Therefore, these features play a crucial role in determining the class of the plant.

Less Important Features

The following features showed relatively low importance scores:

Average dry weight of the root (ADWR)

Percentage of dry matter for vegetative growth (PDMVG)

Percentage of dry matter for root growth (PDMRG)

These features contributed less to the model’s predictive power. This may be because they provide redundant or less significant information compared to other features. Removing or transforming such features can sometimes improve model performance.

Overall Observation

It was observed that features related to plant growth dynamics (such as height, leaf area, and chlorophyll content) were more influential than static measurements like dry weight percentages. This insight guided the feature engineering process, where new features were created by combining the most important variables.


Based on this analysis, new features such as **health_index and growth_efficiency** were created using the most important features to further enhance model performance.

In [ ]:
# growth efficiency
df['growth_efficiency'] = (
    df['Plant height rate (PHR)'] /
    df['Average leaf area of the plant (ALAP)']
)

# health index (FIXED column names)
df['health_index'] = (
    df['Average  of chlorophyll in the plant (ACHP)'] +
    df['Average number of plant leaves (ANPL)']
) / 2

- growth_efficiency: Shows how efficiently plant grows relative to leaf size
- health_index: Combines chlorophyll and leaf count to represent plant health

In [ ]:
# Drop unnecessary column
if 'Random' in df.columns:
    df = df.drop(['Random'], axis=1)

The 'Random' column was removed because it does not represent any meaningful biological or agricultural property of the plant. It appears to be an artificial or index-like feature that does not contribute to predicting plant health or class. Removing such irrelevant features helps reduce noise and improves model performance.

In [ ]:
import lightgbm as lgb

# Define features and target
X = df.drop('Class', axis=1)
y = df['Class']

# Train model
model = lgb.LGBMClassifier()
model.fit(X, y)

print("Model trained with new features")

LightGBM was used to evaluate the effectiveness of both original and newly engineered features. It is a fast and efficient gradient boosting algorithm that helps identify which features contribute most to prediction. By training the model after feature engineering, we can assess whether the new features improve performance.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

Standardization was applied to ensure that all features are on a similar scale. Since different features (such as plant height, leaf area, and weights) have different units and ranges, scaling prevents bias toward larger-valued features and improves the performance of algorithms like K-Means.

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42)
df['cluster_feature'] = kmeans.fit_predict(X_scaled)

K-Means clustering was used to group plants into clusters based on similar characteristics. These clusters represent hidden patterns in the dataset that are not directly visible through individual features.

The newly created 'cluster_feature' assigns each data point to a cluster, effectively capturing complex relationships among multiple features.

This feature helps the model by providing additional information about the overall group behavior of plants, which can improve classification performance

In [ ]:
df.to_csv('final_feature_engineered_data.csv', index=False)

Explanation

The final dataset, including all engineered features and transformations, was saved as a CSV file. This allows reuse in future phases such as model tuning and evaluation without repeating preprocessing and feature engineering steps.